# 🎵 Project #21: The Crown Jewel - Music Sequence Modeling
**Architect:** Kemal Demirbaş 🏰🚀 | **Project Series:** 21 of 21 (**GRAND FINALE**)

[![Hugging Face Space](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Live%20Demo-ffcc00)](https://huggingface.co/spaces/Ironside35/Music-Sequence-Predictor)
[![Framework](https://img.shields.io/badge/DL%20Framework-TensorFlow%20%2F%20Keras-red)](https://www.tensorflow.org/)
[![Architecture](https://img.shields.io/badge/Architecture-LSTM%20(RNN)-blue)](https://en.wikipedia.org/wiki/Long_short-term_memory)
[![Status](https://img.shields.io/badge/Status-Completed-brightgreen)](https://github.com/Ironside35)

---

### 📖 Project Vision: The Sequential Vibe
> *"Music isn't just a list of sounds; it's a journey. A static recommendation is a snapshot, but a sequence recommendation is the whole movie."*

**Project Overview:** For the final entry in the 21-project series, we move beyond simple similarity. This engine uses **Deep Learning (LSTM)** to analyze the sequential relationship between audio features. By understanding the "flow" of a session, it predicts the next track's DNA (Energy, Tempo, Valence) to create a professional DJ-level transition experience.

---

### 🧠 The Brain: Recurrent Neural Architecture (LSTM)
Unlike standard models, **Long Short-Term Memory (LSTM)** networks have a "Memory Cell." This allows the model to remember the mood of the first song while processing the fifth.

The core logic is the **Forget Gate**, which decides which previous "vibes" to keep or discard to ensure a natural progression:

$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$

---

### 💻 The Final Code: Deep Learning Implementation

In [15]:
import pandas as pd
import numpy as np
import zipfile
import os
import joblib
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [20]:
# ==========================================
# 1. EXTRACT DATA AND FIND THE *CORRECT* CSV
# ==========================================
import pandas as pd
import zipfile
import os

zip_path = "/content/Spotify dataset.zip"
extract_dir = "/content/spotify_data_v3"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Find all CSVs and list them
csv_files = []
for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print(f"📦 ZIP'in İçinden Çıkan CSV Dosyaları:\n{csv_files}\n")

# TARGET LOCKING: Select only the file containing the original songs!
correct_csv = csv_files[0] # Yedek
for file in csv_files:
    # Generally, the actual data is in 'data.csv' or 'tracks.csv'
    if "data.csv" in file.lower() or "tracks.csv" in file.lower():
        correct_csv = file
        break

df = pd.read_csv(correct_csv)
print(f"✅ 1. DOĞRU VERİ OKUNDU: {correct_csv} (Satır Sayısı: {len(df)})")
print(f"🔍 Orijinal Kolonlar: {df.columns.tolist()[:10]}...")

📦 ZIP'in İçinden Çıkan CSV Dosyaları:
['/content/spotify_data_v3/data/data_w_genres.csv', '/content/spotify_data_v3/data/data.csv', '/content/spotify_data_v3/data/data_by_genres.csv', '/content/spotify_data_v3/data/data_by_year.csv', '/content/spotify_data_v3/data/data_by_artist.csv']

✅ 1. DOĞRU VERİ OKUNDU: /content/spotify_data_v3/data/data.csv (Satır Sayısı: 170653)
🔍 Orijinal Kolonlar: ['valence', 'year', 'acousticness', 'artists', 'danceability', 'duration_ms', 'energy', 'explicit', 'id', 'instrumentalness']...


In [21]:
# ==========================================
# 2. BULLETPROOF COLUMN FIX (STANDARDIZATION)
# ==========================================
# Find the song name column regardless of its format and force it to 'name'
name_candidates = ['track_name', 'Name', 'song_name', 'title']
for col in name_candidates:
    if col in df.columns:
        df.rename(columns={col: 'name'}, inplace=True) # Changed 'Name' to 'name' for consistency
        break

artist_candidates = ['Artist', 'artists', 'track_artist']
for col in artist_candidates:
    if col in df.columns:
        df.rename(columns={col: 'artists'}, inplace=True)
        break

# Drop corrupted rows without a name from the dataset
# Only attempt to dropna on 'name' if the column exists after renaming
if 'name' in df.columns:
    df = df.dropna(subset=['name'])
else:
    print("Warning: 'name' column not found after renaming, skipping dropping rows based on 'name'.")

print(f"✅ 2. Names Standardized. Columns Secured.")

✅ 2. Names Standardized. Columns Secured.


In [22]:
# ==========================================
# 3. DATA PREP FOR AI (AUDIO DNA)
# ==========================================
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df[features])

def create_sequences(data, seq_length=5):
    x, y = [], []
    for i in range(len(data) - seq_length):
        x.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(x), np.array(y)

X, y = create_sequences(df_scaled)
print(f"✅ 3. Music Vectors (Sequences) Generated.")

✅ 3. Music Vectors (Sequences) Generated.


In [23]:
# ==========================================
# 4. TRAIN THE BRAIN (LSTM NEURAL NETWORK)
# ==========================================
print("\n🧠 4. TRAINING THE BRAIN... (This might take a while)\n")
model = Sequential([
    LSTM(64, input_shape=(X.shape[1], X.shape[2]), return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(len(features))
])
model.compile(optimizer='adam', loss='mse')

# Keeping epochs at 10 for faster results, it will perform well enough
model.fit(X, y, epochs=10, batch_size=64, validation_split=0.2, verbose=1)


🧠 4. TRAINING THE BRAIN... (This might take a while)



/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
2134/2134 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - loss: 0.0491 - val_loss: 0.0467
Epoch 2/10
2134/2134 ━━━━━━━━━━━━━━━━━━━━ 41s 10ms/step - loss: 0.0440 - val_loss: 0.0467
Epoch 3/10
2134/2134 ━━━━━━━━━━━━━━━━━━━━ 40s 10ms/step - loss: 0.0437 - val_loss: 0.0464
Epoch 4/10
2134/2134 ━━━━━━━━━━━━━━━━━━━━ 21s 10ms/step - loss: 0.0435 - val_loss: 0.0464
Epoch 5/10
2134/2134 ━━━━━━━━━━━━━━━━━━━━ 19s 9ms/step - loss: 0.0434 - val_loss: 0.0464
Epoch 6/10
2134/2134 ━━━━━━━━━━━━━━━━━━━━ 22s 10ms/step - loss: 0.0434 - val_loss: 0.0464
Epoch 7/10
2134/2134 ━━━━━━━━━━━━━━━━━━━━ 21s 10ms/step - loss: 0.0433 - val_loss: 0.0463
Epoch 8/10
2134/2134 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - loss: 0.0433 - val_loss: 0.0462
Epoch 9/10
2134/2134 ━━━━━━━━━━━━━━━━━━━━ 24s 11ms/step - loss: 0.0433 - val_loss: 0.0463
Epoch 10/10
2134/2134 ━━━━━━━━━━━━━━━━━━━━ 22s 10ms/step - loss: 0.0432 - val_loss: 0.0464


In [24]:
# ==========================================
# 5. PACKAGE FILES FOR HUGGING FACE
# ==========================================
# 1. Save the model (.h5 for compatibility)
model.save('music_lstm_model.h5')

# 2. Save the scaler
joblib.dump(scaler, 'music_scaler.pkl')

# 3. Save the mini dataset making absolutely sure it has the 'name' column
export_cols = ['name', 'artists'] + features
mini_music = df[export_cols].drop_duplicates(subset=['name']).head(10000)
mini_music.to_csv('music_mini.csv', index=False)

print("\n🏆 GRAND VICTORY! ALL OPERATIONS COMPLETED FLAWLESSLY. 🏆")
print("👉 NOW DOWNLOAD THESE 3 FILES FROM THE LEFT PANEL: 'music_lstm_model.h5', 'music_scaler.pkl', 'music_mini.csv'")


🏆 GRAND VICTORY! ALL OPERATIONS COMPLETED FLAWLESSLY. 🏆
👉 NOW DOWNLOAD THESE 3 FILES FROM THE LEFT PANEL: 'music_lstm_model.h5', 'music_scaler.pkl', 'music_mini.csv'


# 🎵 Project #21: The Crown Jewel - Music Sequence Predictor
**Architect:** Kemal Demirbaş 🏰🚀 | **Project Series:** 21 of 21 (**GRAND FINALE**)

[![Hugging Face Space](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Live%20Demo-ffcc00)](https://huggingface.co/spaces/Ironside35/Music-Sequence-Predictor)
[![Framework](https://img.shields.io/badge/DL%20Framework-TensorFlow%20%2F%20Keras-red)](https://www.tensorflow.org/)
[![Model](https://img.shields.io/badge/Architecture-LSTM%20RNN-blue)](https://en.wikipedia.org/wiki/Long_short-term_memory)

---

### 👑 Project Vision: The Grand Finale
> *"Music is not a static point; it's a continuous stream of energy. To predict the next track, you must understand the history of the rhythm."*

**The Crown Jewel** is the 21st and final project in this series. It represents the pinnacle of Data Science and Deep Learning by utilizing **Recurrent Neural Networks (RNN)**—specifically **LSTM (Long Short-Term Memory)**—to model the sequential flow of music.

---

## 🧠 The Brain: Deep Learning Sequence Logic
While Project #20 used static similarity, Project #21 uses **Temporal Intelligence**:

* **Sequential Memory:** The LSTM architecture tracks transitions across a sliding window of 5 songs.
* **Vector Prediction:** Outputs an "Ideal Next Feature Vector" mapped to the Spotify database.

---

## 🚀 Live Music Discovery Platform
Try the final engine here:
👉 **[Hugging Face Live Demo: Music Sequence Predictor](https://huggingface.co/spaces/Ironside35/Rhythm-Sequence-AI)**

---
## 🏆 THE JOURNEY IS COMPLETE
From data cleaning in Project #1 to Deep Learning Sequence Modeling in Project #21.
**Architect Kemal Demirbaş has officially mastered the 21-fortress marathon.** 🏰🚀👑